# Лаба 1.07

### Математические функции и XML

In [38]:
from meas import *

from math import *
import numpy as np

from pprint import pprint
from lxml import etree              

# Парсим XML
dom = etree.parse('latex.xml')
xml = dom.getroot()
next = lambda x: x.pop(0).text

### Результаты измерений

In [39]:
# Инструментальные погрешности
delta_timer = 0.001   # c
delta_trammel = 0.01  # см
delta_ruler = 0.1     # см

# Расстояние h
direct_h = Measurement(
    36.7, delta_ruler, dim=0.01, direct=True,
    name='h', char='h', unit='\\text{м}')
# Расстояние h1
p = 42
direct_h1 = DirectMultipleMeasurement(
    [p - 10.3, p - 9.8, p - 10.4, p - 9.5, p - 9.6], delta_ruler, dim=0.01,
    name='h1', char='h_1', unit='\\text{м}')
# Время спуска
direct_t = DirectMultipleMeasurement(
    [1.584, 1.614, 1.605, 1.530, 1.730], delta_timer,
    name='t', char='t', unit='\\text{с}')

# Диаметр оси
direct_d = DirectMultipleMeasurement(
    [0.80, 0.79, 0.80], delta_trammel, dim=0.01,
    name='d', char='d', unit='\\text{м}')
# Диаметр маховика
direct_D = DirectMultipleMeasurement(
    [9.94, 9.95, 9.96], delta_trammel, dim=0.01,
    name='D', char='D', unit='\\text{м}')

# Масса диска
direct_m_disc = Measurement(
    208, 1, dim=0.001, direct=True,
    name='m диска', char='m_\\text{диска}', unit='\\text{кг}')
# Масса стержня
direct_m_rod = Measurement(
    75, 1, dim=0.001, direct=True,
    name='m стержня', char='m_\\text{стержня}', unit='\\text{кг}')

# Ускорение свободного падения
g = 9.8195

### Задания

##### Множественные прямые измерения

In [40]:
print("Расстояние h =", direct_h)

print("Расстояние h1 =", direct_h1)
print("Время спуска t =", direct_t)

print("Диаметр оси d =", direct_d)
print("Диаметр маховика D =", direct_D)

Расстояние h = 0.3670±0.0010, ε = 0.27% (0.367, Δ = 0.001000000, ε = 0.272479564)
Расстояние h1 = 0.321±0.005, ε = 1.6% (0.3208, Δ = 0.005124188, ε = 1.597315394)
Время спуска t = 1.61±0.09, ε = 6.0% (1.6126, Δ = 0.091115216, ε = 5.650205631)
Диаметр оси d = 0.00797±0.00016, ε = 2.0% (0.007966666667, Δ = 0.000158079, ε = 1.984252000)
Диаметр маховика D = 0.09950±0.00026, ε = 0.26% (0.0995, Δ = 0.000257056, ε = 0.258347720)


##### Простые косвенные измерения

$m = m_\text{диска} + m_\text{стержня}$

$r = \frac{d}{2}$

$R = \frac{D}{2}$

In [41]:
r = direct_d / 2
r.naming('r', '\\text{м}')
print("Радиус оси r =", r)

R = direct_D / 2
R.naming('R', '\\text{м}')
print("Радиус маховика R =", R)

m = direct_m_disc + direct_m_rod
m.naming('m', '\\text{кг}')
print("Масса маятника m =", m)

Радиус оси r = 0.00398±0.00008, ε = 2.0% (0.003983333333, Δ = 0.000079039, ε = 1.984252000)
Радиус маховика R = 0.04975±0.00013, ε = 0.26% (0.04975, Δ = 0.000128528, ε = 0.258347720)
Масса маятника m = 0.2830±0.0009, ε = 0.33% (0.283, Δ = 0.000942809, ε = 0.333148071)


##### Момент инерции $I$ по формулам (6) и (11)

$I = m r ^ { 2 } \left( \frac { g t ^ { 2 } } { 2 h } - 1 \right)$ (6)

$I = m r ^ { 2 } \Biggl ( \frac { g t ^ { 2 } } { h } \cdot \frac { h _ { 1 } } { h + h _ { 1 } } - 1 \Biggr )$ (11)

$\frac{\Delta I}{I} = \sqrt{\left(\frac{\Delta m}{m}\right)^2 + \left(2\frac{\Delta r}{r}\right)^2 + \left(\frac{\Delta A}{A}\right)^2}$,
$A = \frac{gt^2}{h} \cdot \frac{h_1}{h + h_1} - 1$, 
$\frac{\Delta A}{A} = \sqrt{\left(2\frac{\Delta t}{t}\right)^2 +\left(\frac{\Delta h}{h}\right)^2 +\left(\frac{\Delta h_1}{h_1}\right)^2 }$

$\frac{h_1}{h_1 + h} < \frac{1}{2}$

In [42]:
vA6 = (g * direct_t.value_**2)/(2 * direct_h.value_) - 1
eA6 = sqrt((2*direct_t.epsilon_/100)**2 + (direct_h.epsilon_/100)**2)

vI6 = m.value_ * r.value_**2 * vA6
eI6 = sqrt((m.epsilon_/100)**2 + (2 * r.epsilon_/100)**2 + eA6**2) * 100

vAx = (g * direct_t.value_**2)/(direct_h.value_) * (direct_h1.value_)/(direct_h.value_ + direct_h1.value_) - 1
eAx = sqrt((2*direct_t.epsilon_/100)**2 + (direct_h.epsilon_/100)**2 + (direct_h1.epsilon_/100)**2)

vIx = m.value_ * r.value_**2 * vAx
eIx = sqrt((m.epsilon_/100)**2 + (2 * r.epsilon_/100)**2 + eAx**2) * 100

I6 = Measurement(vI6, epsilon=eI6, char='I', unit='\\text{кг}\\cdot\\text{м}^2')
Ix = Measurement(vIx, epsilon=eIx, char='I', unit='\\text{кг}\\cdot\\text{м}^2')

print("По формуле 6:  I =", I6)
print("По формуле 11: I =", Ix)

print(direct_h1.value_ / (direct_h1.value_ + direct_h.value_))


По формуле 6:  I = 0.000152±0.000018, ε = 12.0% (0.00015172596, Δ = 0.000018184, ε = 11.984721584)
По формуле 11: I = 0.000141±0.000017, ε = 12.0% (0.000141232803, Δ = 0.000017076, ε = 12.090697578)
0.4664146554230881


$I_\text{разм} = \frac{\rho\pi}{2} \left( L_{\text{диска}} R^4_{\text{диска}} + 2L_{\text{сплош}} r^4 + 2L_{\text{трубки}} \left( r^4_{\text{внешн}} - r^4_{\text{внутр}} \right) \right)$

In [43]:
cm, mm = 0.01, 0.001 
L_tr = Measurement(3.4, 0.1, dim=cm, name='L трубки', char='L_\\text{трубки}', unit='\\text{м}')
L_so = Measurement(5.6, 0.1, dim=cm, name='L сплош', char='L_\\text{сплош}', unit='\\text{м}')
r_in = Measurement(3.50, 0.05, dim=mm, name='r внутр', char='r_\\text{внутр}', unit='\\text{м}')
r_out = Measurement(5.00, 0.05, dim=mm, name='r внешн', char='r_\\text{внеш}', unit='\\text{м}')
L_ds = Measurement(8.00, 0.05, dim=mm, name='L диска', char='L_\\text{диска}', unit='\\text{м}')
R_ds = Measurement(43.00, 0.05, dim=mm, name='R диска', char='R_\\text{диска}', unit='\\text{м}')
rho  = Measurement(2745, 10, name='Плотность', char='\\rho', unit='\\frac{\\text{кг}}{\\text{м}^3}')

D = Measurement(r_out.value_**4 - r_in.value**4, (2/3)*sqrt((4*r_out.value_**3*r_out.delta_)**2 + (4*r_in.value_**3*r_in.delta_)**2))
print('D', D)
C = Measurement(2*L_tr.value_*D.value_, epsilon=sqrt(((2/3)*L_tr.delta_/L_tr.value_)**2 + (D.delta_/D.value_)**2))
print('C', C)
B = Measurement(2*L_so.value_*r_out.value_**4, epsilon=sqrt(((2/3)*L_so.delta_/L_so.value_)**2 + ((2/3)*4*r_out.delta_/r_out.value_)**2))
print('B', B)
A = Measurement(L_ds.value_*R_ds.value_**4, epsilon=sqrt(((2/3)*L_ds.delta_/L_ds.value_)**2 + ((2/3)*4*R.delta_/R.value_)**2))
print('A', A)
S = Measurement(A.value_+B.value_+C.value_, sqrt(A.delta_**2+B.delta_**2+C.delta_**2))
print('S', S)
I_dim = Measurement(
    rho.value_*pi/2 * (L_ds.value_*R_ds.value_**4 + 2*L_so.value_*r_out.value_**4 + 2*L_tr.value_*(r_out.value_**4 - r_in.value_**4)),
    epsilon=sqrt(((2/3)*rho.delta_/rho.value_)**2 + (S.delta_/S.value_)**2),
    char='I_\\text{разм}', unit='\\text{кг}\\cdot\\text{м}^2'
)
print(I_dim)

D 0.000000000475±0.000000000018, ε = 3.7% (4.75e-10, Δ = 0.000000000, ε = 3.709923930)
C 0.000000000032296±0.000000000000014, ε = 0.04% (3.2e-11, Δ = 0.000000000, ε = 0.041962139)
B 0.000000000070000±0.000000000000020, ε = 0.029% (7e-11, Δ = 0.000000000, ε = 0.029203330)
A 0.0000000273504±0.0000000000022, ε = 0.008% (2.735e-08, Δ = 0.000000000, ε = 0.008051285)
S 0.0000000274527±0.0000000000022, ε = 0.008% (2.7453e-08, Δ = 0.000000000, ε = 0.008021781)
0.0001183716±0.0000000029, ε = 0.0024% (0.000118371554, Δ = 0.000000003, ε = 0.002429983)


In [44]:
print("По формуле 6:  I =", I6)
print("По формуле 11: I =", Ix)
print("По размерам:   I =", I_dim)

По формуле 6:  I = 0.000152±0.000018, ε = 12.0% (0.00015172596, Δ = 0.000018184, ε = 11.984721584)
По формуле 11: I = 0.000141±0.000017, ε = 12.0% (0.000141232803, Δ = 0.000017076, ε = 12.090697578)
По размерам:   I = 0.0001183716±0.0000000029, ε = 0.0024% (0.000118371554, Δ = 0.000000003, ε = 0.002429983)


### Latex

In [45]:
print(L_tr.value_latex_m(2), L_so.value_latex_m(2), r_in.value_latex_m(3), 
      r_out.value_latex_m(3), L_ds.value_latex_m(3), R_ds.value_latex_m(2), 
      rho.value_latex_m(), sep='\n')

$L_\text{трубки} = (3.4\pm0.1)\cdot 10^{-2} \text{м}$
$L_\text{сплош} = (5.6\pm0.1)\cdot 10^{-2} \text{м}$
$r_\text{внутр} = (3.50\pm0.05)\cdot 10^{-3} \text{м}$
$r_\text{внеш} = (5.00\pm0.05)\cdot 10^{-3} \text{м}$
$L_\text{диска} = (8.00\pm0.05)\cdot 10^{-3} \text{м}$
$R_\text{диска} = (4.300\pm0.005)\cdot 10^{-2} \text{м}$
$\rho = (2745\pm10) \frac{\text{кг}}{\text{м}^3}$


In [46]:
t1 = xml.findall('t1')
table1 = next(t1) + direct_h.latex_m(2) + next(t1)

for h1, t in zip(direct_h1.measurments, direct_t.measurments):
    table1 += h1.latex_m(2) + next(t1) + t.latex_m() + next(t1)
    
print(table1)


\begin{table}[H]
\centering
\caption{Результаты прямых измерений}
\label{tab:meas7}
\begin{tabular}{|c|c|c|c|}
\hline
№ & $h, \text{см}$ & $h_1, \text{см}$ & $t, \text{с}$ \\ \hline
1 & \multirow{5}{*}{$36.7\pm0.1$} & $31.7\pm0.1$ & $1.584\pm0.001$ \\ \cline{1-1} \cline{3-4}
2 & & $32.2\pm0.1$ & $1.614\pm0.001$ \\ \cline{1-1} \cline{3-4}
3 & & $31.6\pm0.1$ & $1.605\pm0.001$ \\ \cline{1-1} \cline{3-4}
4 & & $32.5\pm0.1$ & $1.530\pm0.001$ \\ \cline{1-1} \cline{3-4}
5 & & $32.4\pm0.1$ & $1.730\pm0.001$ \\ \hline
\end{tabular}
\end{table}



In [47]:
print(', '.join([f'$({d.latex(3)}) \\text{{мм}}$' for d in direct_d.measurments]))
print(', '.join([f'$({d.latex(3)}) \\text{{мм}}$' for d in direct_D.measurments]))

$(8.0\pm0.1) \text{мм}$, $(7.9\pm0.1) \text{мм}$, $(8.0\pm0.1) \text{мм}$
$(99.4\pm0.1) \text{мм}$, $(99.5\pm0.1) \text{мм}$, $(99.6\pm0.1) \text{мм}$


In [48]:
print(direct_h1.meas_latex(), direct_t.meas_latex(), direct_d.meas_latex(), direct_D.meas_latex(), sep='\n\n')

\centerline{$h_1 = (0.321\pm0.005) \text{м}, \quad \varepsilon = 1.6 \%, \quad \alpha = 0.95$}

\centerline{$t = (1.61\pm0.09) \text{с}, \quad \varepsilon = 6.0 \%, \quad \alpha = 0.95$}

\centerline{$d = (0.00797\pm0.00016) \text{м}, \quad \varepsilon = 2.0 \%, \quad \alpha = 0.95$}

\centerline{$D = (0.09950\pm0.00026) \text{м}, \quad \varepsilon = 0.26 \%, \quad \alpha = 0.95$}


In [49]:
print(f"$m = {direct_m_disc.value} + {direct_m_rod.value} = {m.value_latex(hide_char=True)}$")

print(f"$r = \\frac{{{direct_d.value}}}{{2}} = {r.value_latex(hide_char=True)}$")

print(f"$R = \\frac{{{direct_D.value}}}{{2}} = {R.value_latex(hide_char=True)}$")

$m = 0.208 + 0.075 = (0.2830\pm0.0009) \text{кг}$
$r = \frac{0.00797}{2} = (0.00398\pm0.00008) \text{м}$
$R = \frac{0.0995}{2} = (0.04975\pm0.00013) \text{м}$


In [50]:
print(f"""$I =
{m.value} \\cdot {{{r.value}}}^{{ 2 }} \\cdot 
\\left( \\frac {{ {g} {{{direct_t.value}}} ^ {{ 2 }} }} {{ 2 \\cdot {direct_h.value} }} - 1 \\right) =
({I6.latex()}) {I6._unit}$""".replace('\n', ' '))

print(f"""$I =
{m.value} \\cdot {{{r.value}}}^{{ 2 }} \\cdot 
\\left( \\frac {{ {g} {{{direct_t.value}}} ^ {{ 2 }} }} {{ 2 \\cdot {direct_h.value} }} 
\\cdot \\frac{{{direct_h1.value}}} {{ {direct_h.value} + {direct_h1.value} }} - 1 \\right) =
({Ix.latex()}) {Ix._unit}$""".replace('\n', ' '))

print(f"""$I_\\text{{разм}} = 
\\frac{{{rho.value} \\cdot \\pi}}{{2}} \\left( {L_ds.value} \\cdot {R_ds.value}^4 + 2 \\cdot {L_so.value} \\cdot {r_in.value}^4 + 
2 \\cdot {L_tr.value} \\left( {r_out.value}^4 - {r_in.value}^4 \\right) \\right)$
""".replace('\n', ' '))

print(I_dim.value_latex_m())

$I = 0.283 \cdot {0.00398}^{ 2 } \cdot  \left( \frac { 9.8195 {1.61} ^ { 2 } } { 2 \cdot 0.367 } - 1 \right) = (0.00152\pm0.00018) \text{кг}\cdot\text{м}^2$
$I = 0.283 \cdot {0.00398}^{ 2 } \cdot  \left( \frac { 9.8195 {1.61} ^ { 2 } } { 2 \cdot 0.367 }  \cdot \frac{0.321} { 0.367 + 0.321 } - 1 \right) = (0.00141\pm0.00017) \text{кг}\cdot\text{м}^2$
$I_\text{разм} =  \frac{2745 \cdot \pi}{2} \left( 0.008 \cdot 0.043^4 + 2 \cdot 0.056 \cdot 0.0035^4 +  2 \cdot 0.034 \left( 0.005^4 - 0.0035^4 \right) \right)$ 
$I_\text{разм} = (0.0001183716\pm0.0000000029) \text{кг}\cdot\text{м}^2$


In [51]:
print(direct_t.DDM(), direct_h1.DDM(), direct_d.DDM(), direct_D.DDM(), sep='\n\n\n')

$\bar{t} = 1.61 \text{с}$.

\centerline{$\sigma_{\bar{t}} = 0.032774380238228755, \quad \Delta_{\bar{t}} = 2.78 \cdot 0.032774380238228755 = 0.09111277706227593 \text{с}$}

\centerline{$\Delta_{t} = \sqrt{(0.09111277706227593)^2 + (\frac23 \cdot 0.001)^2} = 0.09 \text{с}, \quad \varepsilon_{t} = \frac{0.09}{1.61} \cdot 100\% = 6.0\%$}


$\bar{h_1} = 0.321 \text{м}$.

\centerline{$\sigma_{\bar{h_1}} = 0.0018275666882497082, \quad \Delta_{\bar{h_1}} = 2.78 \cdot 0.0018275666882497082 = 0.005080635393334188 \text{м}$}

\centerline{$\Delta_{h_1} = \sqrt{(0.005080635393334188)^2 + (\frac23 \cdot 0.001)^2} = 0.005 \text{м}, \quad \varepsilon_{h_1} = \frac{0.005}{0.321} \cdot 100\% = 1.6\%$}


$\bar{d} = 0.00797 \text{м}$.

\centerline{$\sigma_{\bar{d}} = 3.333333333333313e-05, \quad \Delta_{\bar{d}} = 4.3 \cdot 3.333333333333313e-05 = 0.00014333333333333247 \text{м}$}

\centerline{$\Delta_{d} = \sqrt{(0.00014333333333333247)^2 + (\frac23 \cdot 0.0001)^2} = 0.00016 \text{м}, \quad \varepsilon

In [52]:
print(R.delta, r.delta)

0.00013 8e-05


$I = m r ^ { 2 } \left( \frac { g t ^ { 2 } } { 2 h } - 1 \right)$ (6)

$I = m r ^ { 2 } \Biggl ( \frac { g t ^ { 2 } } { h } \cdot \frac { h _ { 1 } } { h + h _ { 1 } } - 1 \Biggr )$ (11)

$\frac{\Delta I}{I} = \sqrt{\left(\frac{\Delta m}{m}\right)^2 + \left(2\frac{\Delta r}{r}\right)^2 + \left(\frac{\Delta A}{A}\right)^2}$,
$A = \frac{gt^2}{h} \cdot \frac{h_1}{h + h_1} - 1$, 
$\frac{\Delta A}{A} = \sqrt{\left(2\frac{\Delta t}{t}\right)^2 +\left(\frac{\Delta h}{h}\right)^2 +\left(\frac{\Delta h_1}{h_1}\right)^2 }$

In [53]:
print(f"""$A = \\frac{{ {g} \\cdot {t.value}^2}}{{2 \\cdot {direct_h.value}}} - 1 = {vA6}$, 

$\\frac{{\\Delta A}}{{A}} = \\sqrt{{\\left(2\\cdot\\frac{{{direct_t.delta}}}{{{direct_t.value}}}\\right)^2 + \\left(\\frac{{{direct_h.delta}}}{{{direct_h.value}}}\\right)^2 }} = {eA6}$,

$\\frac{{\\Delta I}}{{I}} = \\sqrt{{\\left(\\frac{{{m.delta}}}{{{m.value}}}\\right)^2 + \\left(2\\frac{{{r.delta}}}{{{r.value}}}\\right)^2 + \\left({eA6}\\right)^2}} = {eI6}$

\\centerline{{$\\Delta_I = {I6.delta} {Ix._unit}, \\quad \\varepsilon_I = {I6.epsilon} \\%$}}""")

$A = \frac{ 9.8195 \cdot 1.73^2}{2 \cdot 0.367} - 1 = 33.7893749098365$, 

$\frac{\Delta A}{A} = \sqrt{\left(2\cdot\frac{0.09}{1.61}\right)^2 + \left(\frac{0.001}{0.367}\right)^2 } = 0.11303695847559815$,

$\frac{\Delta I}{I} = \sqrt{\left(\frac{0.0009}{0.283}\right)^2 + \left(2\frac{8e-05}{0.00398}\right)^2 + \left(0.11303695847559815\right)^2} = 11.984721584299988$

\centerline{$\Delta_I = 1.8e-05 \text{кг}\cdot\text{м}^2, \quad \varepsilon_I = 12.0 \%$}


In [54]:
print(f"""$A = \\frac{{ {g} \\cdot {t.value}^2}}{{{direct_h.value}}} \\cdot \\frac{{{direct_h1.value}}}{{{direct_h.value} + {direct_h1.value}}} - 1 = {vAx}$, 

$\\frac{{\\Delta A}}{{A}} = \\sqrt{{\\left(2\\cdot\\frac{{{direct_t.delta}}}{{{direct_t.value}}}\\right)^2 + \\left(\\frac{{{direct_h.delta}}}{{{direct_h.value}}}\\right)^2 +\\left(\\frac{{{direct_h1.delta}}}{{{direct_h1.value}}}\\right)^2 }} = {eAx}$,

$\\frac{{\\Delta I}}{{I}} = \\sqrt{{\\left(\\frac{{{m.delta}}}{{{m.value}}}\\right)^2 + \\left(2\\frac{{{r.delta}}}{{{r.value}}}\\right)^2 + \\left({eAx}\\right)^2}} = {eIx}$

\\centerline{{$\\Delta_I = {Ix.delta} {Ix._unit}, \\quad \\varepsilon_I = {Ix.epsilon} \\%$}}""")

$A = \frac{ 9.8195 \cdot 1.73^2}{0.367} \cdot \frac{0.321}{0.367 + 0.321} - 1 = 31.452548621912037$, 

$\frac{\Delta A}{A} = \sqrt{\left(2\cdot\frac{0.09}{1.61}\right)^2 + \left(\frac{0.001}{0.367}\right)^2 +\left(\frac{0.005}{0.321}\right)^2 } = 0.1141599563253143$,

$\frac{\Delta I}{I} = \sqrt{\left(\frac{0.0009}{0.283}\right)^2 + \left(2\frac{8e-05}{0.00398}\right)^2 + \left(0.1141599563253143\right)^2} = 12.090697577932454$

\centerline{$\Delta_I = 1.7e-05 \text{кг}\cdot\text{м}^2, \quad \varepsilon_I = 12.0 \%$}


In [55]:
print(I6.meas_latex(4))
print(Ix.meas_latex(4))
print(I_dim.meas_latex(4))

\centerline{$I = (1.52\pm0.18)\cdot 10^{-4} \text{кг}\cdot\text{м}^2, \quad \varepsilon = 12.0 \%, \quad \alpha = 0.95$}
\centerline{$I = (1.41\pm0.17)\cdot 10^{-4} \text{кг}\cdot\text{м}^2, \quad \varepsilon = 12.0 \%, \quad \alpha = 0.95$}
\centerline{$I_\text{разм} = (1.183716\pm0.000029)\cdot 10^{-4} \text{кг}\cdot\text{м}^2, \quad \varepsilon = 0.0024 \%, \quad \alpha = 0.95$}
